# Pima diabetes

One of two datasets called "diabetes" (the other, [Diabetes progression](deep-dives/diabetes_progression.ipynb), is a deep dive). This one is a classification, and its surprise is that the values are in their units, but some of them are not values at all.

## Reading the dataset first

OpenML, an open repository of datasets for machine learning, publishes it under the name `diabetes`; scikit-learn downloads it the first time:

In [ ]:
from sklearn.datasets import fetch_openml

pima = fetch_openml(name="diabetes", version=1, as_frame=True)
pima.frame.head()

,preg,plas,pres,skin,insu,mass,pedi,age,class
0,6,148,72,35,0,33.6,0.627,50,tested_positive
1,1,85,66,29,0,26.6,0.351,31,tested_negative
2,8,183,64,0,0,23.3,0.672,32,tested_positive
3,1,89,66,23,94,28.1,0.167,21,tested_negative
4,0,137,40,35,168,43.1,2.288,33,tested_positive


The column names are abbreviations, so the documentation is needed before anything else. Its essential part:

In [ ]:
description = pima.DESCR
print(description[description.index("Past Usage"):description.index("Relevant Information")].strip())
print(description[description.index("Relevant Information"):description.index("Number of Instances")].strip())
print(description[description.index("For Each Attribute"):description.index("9. Class Distribution")].strip())

Past Usage:
     1. Smith,~J.~W., Everhart,~J.~E., Dickson,~W.~C., Knowler,~W.~C., &
        Johannes,~R.~S. (1988). Using the ADAP learning algorithm to forecast
        the onset of diabetes mellitus.  In {it Proceedings of the Symposium
        on Computer Applications and Medical Care} (pp. 261--265).  IEEE
        Computer Society Press.
 
        The diagnostic, binary-valued variable investigated is whether the
        patient shows signs of diabetes according to World Health Organization
        criteria (i.e., if the 2 hour post-load plasma glucose was at least 
        200 mg/dl at any survey  examination or if found during routine medical
        care).   The population lives near Phoenix, Arizona, USA.
 
        Results: Their ADAP algorithm makes a real-valued prediction between
        0 and 1.  This was transformed into a binary decision using a cutoff of 
        0.448.  Using 576 training instances, the sensitivity and specificity
        of their algorithm was 76% on 

Three facts from it decide the workflow:

- **Who is in the data:** 768 women, at least 21 years old, of Pima heritage, a Native American people of Arizona, studied by the US National Institute of Diabetes and Digestive and Kidney Diseases. A model learned here describes this population; nothing in the data says it would hold for anyone else.
- **What the target is:** whether the woman showed signs of diabetes according to the World Health Organization's criteria. A diagnosis, yes or no, made at any examination of the study (the original study used it to forecast the onset of diabetes): a classification, unlike [Diabetes progression](deep-dives/diabetes_progression.ipynb).
- **What the features are:** number of pregnancies (`preg`), plasma glucose two hours after a glucose test (`plas`), diastolic blood pressure (`pres`), triceps skin-fold thickness (`skin`), insulin two hours after the test (`insu`), body mass index (`mass`), a score of diabetes in the family (`pedi`) and age.

In [ ]:
X = pima.data
y = (pima.target == "tested_positive").astype(int).rename("diabetes")
y.value_counts()

diabetes
0    500
1    268
Name: count, dtype: int64

The documentation also says there are no missing values. The data disagrees:

In [ ]:
X.describe().round(1)

,preg,plas,pres,skin,insu,mass,pedi,age
count,768.0,768.0,768.0,768.0,768.0,768.0,768.0,768.0
mean,3.8,120.9,69.1,20.5,79.8,32.0,0.5,33.2
std,3.4,32.0,19.4,16.0,115.2,7.9,0.3,11.8
min,0.0,0.0,0.0,0.0,0.0,0.0,0.1,21.0
25%,1.0,99.0,62.0,0.0,0.0,27.3,0.2,24.0
50%,3.0,117.0,72.0,23.0,30.5,32.0,0.4,29.0
75%,6.0,140.2,80.0,32.0,127.2,36.6,0.6,41.0
max,17.0,199.0,122.0,99.0,846.0,67.1,2.4,81.0


In [ ]:
measurements = ["plas", "pres", "skin", "insu", "mass"]
(X[measurements] == 0).sum()

plas      5
pres     35
skin    227
insu    374
mass     11
dtype: int64

Glucose, blood pressure, skin fold, insulin and body mass index all have a minimum of 0, and no living person has any of them at 0: these are measurements that were never taken, stored as 0. Insulin is missing for 374 women, almost half; the skin fold for 227. A model given these zeros would treat them as real, very low values, and an insulin of 0 would become evidence about diabetes when all it says is that no one measured it. The documentation's "Missing Attribute Values: None" is true only in the narrowest sense: there are no empty cells, because the gaps were filled with zeros.

## Missing values, without leaking

Turning the impossible zeros into missing values is a fixed rule: it learns nothing from the data, so it can be done once, before the split. *Filling* them is different: the usual fill, the median of each column, is learned from data, so it belongs inside the pipeline, fitted on the training rows only ([the test set stands for production](../ml-fundamentals/overfitting_and_the_split.ipynb#the-test-set-stands-for-production)). Filling them with the median of all 768 women before splitting would be a small leak, but exactly the kind the split exists to prevent. Why values go missing, and the other ways of filling them: [Missing values](../../data-preparation/missing_values.ipynb).

`SimpleImputer(add_indicator=True)` also adds, for each column that had missing values, a 0/1 column saying whether the value was missing. Whether insulin was measured at all may say something about a patient, and the model can then use it.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

X = X.assign(**{column: X[column].replace(0, np.nan) for column in measurements})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.isna().sum()

preg      0
plas      4
pres     23
skin    175
insu    290
mass      9
pedi      0
age       0
dtype: int64

## Comparing models

A baseline that always answers "no diabetes", a logistic regression and a random forest, compared by 5-fold cross-validation on the training rows. The measures are those of [Evaluating a classifier](../evaluation/classification_metrics.ipynb): accuracy, recall of the positive class, and the area under the ROC curve:

In [ ]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

candidates = {
    "always 'no diabetes'": DummyClassifier(),
    "logistic regression": make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(), LogisticRegression()),
    "random forest": make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                                   RandomForestClassifier(min_samples_leaf=5, random_state=42, n_jobs=-1)),
}
cross_validated = pd.DataFrame({
    name: pd.DataFrame(cross_validate(model, X_train, y_train, cv=5, scoring=["accuracy", "recall", "roc_auc"])).mean()
    for name, model in candidates.items()
}).T[["test_accuracy", "test_recall", "test_roc_auc"]]
cross_validated.round(3)

,test_accuracy,test_recall,test_roc_auc
always 'no diabetes',0.651,0.000,0.500
logistic regression,0.783,0.584,0.841
random forest,0.767,0.561,0.830


`cross_validate` calls each held-out fold "test"; it is the validation fold of each round, not the test set, which is still untouched. Answering "no diabetes" every time is right 65% of the time and finds no case at all. Logistic regression is right about 78% of the time, finds about 58% of the cases, and has an AUC of 0.84; the forest is slightly behind on all three, so logistic regression is chosen.

## Choosing a threshold

For screening, a missed case costs more than a second test, so the default threshold of 0.5 is not necessarily the right one ([the threshold](../evaluation/classification_metrics.ipynb#the-threshold)). Like any setting, it is chosen on the training rows: cross-validated probabilities, each predicted by a model that didn't see that row, show the trade-off without touching the test set:

In [ ]:
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import cross_val_predict

chosen_name = cross_validated["test_roc_auc"].idxmax()
final_model = candidates[chosen_name]
training_probabilities = cross_val_predict(final_model, X_train, y_train, cv=5, method="predict_proba")[:, 1]
pd.DataFrame({
    threshold: {
        "precision": precision_score(y_train, training_probabilities >= threshold),
        "recall": recall_score(y_train, training_probabilities >= threshold),
    }
    for threshold in (0.2, 0.3, 0.4, 0.5)
}).T.rename_axis("threshold").round(3)

,precision,recall
threshold,,
0.2,0.528,0.874
0.3,0.594,0.766
0.4,0.670,0.654
0.5,0.740,0.584


At 0.5 the model finds 58% of the cases, and when it flags one it is right 74% of the time. At 0.3 it finds about 77% of them, at a precision of about 59%: for a screening, where a positive leads to a confirmatory test and not to a diagnosis, that is the better trade. The original study reported in 1988 a sensitivity and a specificity of 76% each (quoted in the documentation above): at 0.3 this model finds about the same share of the cases, so it is in the same range, not better. The threshold of 0.3 is chosen here, on the training rows. A model's `predict` still uses 0.5; to ship the threshold with the model, scikit-learn wraps both in one object, `FixedThresholdClassifier(model, threshold=0.3)`, and `TunedThresholdClassifierCV` chooses the threshold by cross-validation.

## Testing once

The model, trained on all the training rows, and the chosen threshold, measured once on the test set:

In [ ]:
from sklearn.metrics import classification_report

final_model.fit(X_train, y_train)
screening_threshold = 0.3
test_flagged = (final_model.predict_proba(X_test)[:, 1] >= screening_threshold).astype(int)
print(classification_report(y_test, test_flagged, target_names=["no diabetes", "diabetes"]))

              precision    recall  f1-score   support

 no diabetes       0.86      0.71      0.78       100
    diabetes       0.59      0.78      0.67        54

    accuracy                           0.73       154
   macro avg       0.72      0.74      0.72       154
weighted avg       0.76      0.73      0.74       154



On the 154 test women, the model with a threshold of 0.3 finds 78% of the cases of diabetes, and 59% of the women it flags have it: what the cross-validation predicted. Two in five flagged women would go on to a confirmatory test they turn out not to need, which is the price, chosen deliberately, of missing fewer than one case in four.

## Two datasets called "diabetes" *(deep dive)*

| | [Diabetes progression](deep-dives/diabetes_progression.ipynb) | Pima diabetes |
| --- | --- | --- |
| Who | 442 patients who already have diabetes | 768 women of Pima heritage, aged 21 or more |
| Target | a measure of progression one year later: regression | signs of diabetes, yes or no: classification |
| Units of the features | removed: centred and scaled, using every row | kept, but zeros stand for missing values |
| What the documentation hides or gets wrong | which sex each code is | it says there are no missing values |
| What a pipeline written without reading would do | work on features scaled with the test rows included, and read weights in units that no longer exist | learn that an insulin level of 0 is real |